# Modern Enterprise Data Stack Notebook

This notebook documents the current project implementation for both **batch** and **streaming** data processing.

It reflects the latest repository structure and operational flow, including:
- Makefile-driven operations (`make up`, `make ps`, `make logs`, `make run-*`)
- Canonical layout (`infra/`, `pipelines/`, `docs/`, `k8s/`, `iac/`, `java-api/`)
- Spark batch + streaming processing with MinIO/PostgreSQL
- Optional Apache Iceberg table writes via `make run-iceberg-demo`
- Monitoring, governance, and ML stubs

## Architecture Overview

Below is a text-based diagram of the pipeline architecture:

```
                              ┌─────────────────────────────┐
                              │         Batch Source        │
                              │   (MySQL, Files, etc.)      │
                              └────────────┬────────────────┘
                                           │
                                           │ (Extract/Validate)
                                           ▼
                           ┌────────────────────────────────┐
                           │      Airflow Batch DAG         │
                           │ - Extracts data from MySQL     │
                           │ - Validates with GreatExpect.  │
                           │ - Uploads raw data to MinIO    │
                           └────────────┬───────────────────┘
                                        │ (spark-submit)
                                        ▼
                           ┌───────────────────────────────────┐
                           │         Spark Batch Job           │
                           │ - Transforms, cleans, enriches    │
                           │ - Writes to PostgreSQL & MinIO    │
                           └────────────┬──────────────────────┘
                                        │
                                        ▼
                           ┌────────────────────────────────┐
                           │       Processed Data Store     │
                           │         (PostgreSQL)           │
                           └────────────────────────────────┘
                                        
Streaming Side:
                              ┌─────────────────────────────┐
                              │       Streaming Source      │
                              │         (Kafka)             │
                              └────────────┬────────────────┘
                                           │
                                           ▼
                           ┌───────────────────────────────────┐
                           │    Spark Streaming Job            │
                           │ - Consumes Kafka messages         │
                           │ - Detects anomalies               │
                           │ - Writes to PostgreSQL & MinIO    │
                           └───────────────────────────────────┘

Monitoring & Governance:
                              ┌────────────────────────────────┐
                              │   Monitoring & Governance      │
                              │ - Prometheus & Grafana         │
                              │ - Apache Atlas/OpenMetadata    │
                              └────────────────────────────────┘

ML & Serving:
                              ┌────────────────────────────────┐
                              │        AI/ML Serving           │
                              │ - Feature Store (Feast)        │
                              │ - MLflow Model Tracking        │
                              │ - BI Dashboards                │
                              └────────────────────────────────┘
```

## Directory Structure

```
modern-enterprise-data-stack/
  ├── README.md
  ├── Makefile
  ├── docs/
  │   ├── QUICK_START.md
  │   ├── ARCHITECTURE.md
  │   ├── DEPLOYMENT_STRATEGIES.md
  │   └── ICEBERG.md
  ├── infra/
  │   ├── compose/
  │   │   ├── docker-compose.yaml
  │   │   └── docker-compose.ci.yaml
  │   ├── dockerfiles/
  │   │   ├── airflow.Dockerfile
  │   │   ├── spark.Dockerfile
  │   │   └── (custom API Dockerfile if needed)
  │   └── README.md
  ├── pipelines/
  │   ├── airflow/dags/
  │   ├── spark/
  │   ├── kafka/
  │   ├── great_expectations/
  │   ├── governance/
  │   ├── monitoring/
  │   ├── ml/
  │   ├── storage/
  │   └── bi_dashboards/
  ├── k8s/
  ├── iac/
  ├── ops/
  ├── java-api/
  ├── notebooks/
  │   └── modern-data-stack.ipynb
  └── web/
```

In [ ]:
# Show current Docker Compose configuration used by the project
from pathlib import Path

compose_path = Path("infra/compose/docker-compose.yaml")
docker_compose_yaml = compose_path.read_text(encoding="utf-8")

print(f"Loaded compose file: {compose_path}")
print("-" * 80)
print(docker_compose_yaml[:6000])
print("\n... (truncated)")

In [ ]:
# Airflow Batch Ingestion DAG (airflow/dags/batch_ingestion_dag.py)
from airflow import DAG
from airflow.operators.python_operator import PythonOperator
from airflow.providers.mysql.hooks.mysql import MySqlHook
from airflow.providers.postgres.hooks.postgres import PostgresHook
from airflow.operators.bash_operator import BashOperator
from datetime import datetime
import pandas as pd
import boto3
import logging
import great_expectations as ge

default_args = {"owner": "airflow", "start_date": datetime(2023, 1, 1), "retries": 1}


def extract_data_from_mysql(**kwargs):
    logging.info("Extracting data from MySQL...")
    mysql_hook = MySqlHook(mysql_conn_id="mysql_default")
    df = mysql_hook.get_pandas_df("SELECT * FROM orders;")
    df.to_csv("/tmp/orders.csv", index=False)
    logging.info(f"Extracted {len(df)} records.")


def validate_data_with_ge(**kwargs):
    logging.info("Validating data with Great Expectations...")
    df = pd.read_csv("/tmp/orders.csv")
    ge_df = ge.from_pandas(df)
    result = ge_df.expect_column_values_to_not_be_null("order_id")
    if not result["success"]:
        raise ValueError("Validation failed: 'order_id' contains null values")
    logging.info("Data validation passed.")


def load_to_minio(**kwargs):
    logging.info("Uploading raw data to MinIO...")
    s3 = boto3.client(
        "s3",
        endpoint_url="http://minio:9000",
        aws_access_key_id="minio",
        aws_secret_access_key="minio123",
        region_name="us-east-1",
    )
    bucket_name = "raw-data"
    try:
        s3.create_bucket(Bucket=bucket_name)
    except Exception as e:
        logging.info(f"Bucket may already exist: {e}")
    s3.upload_file("/tmp/orders.csv", bucket_name, "orders/orders.csv")
    logging.info("File uploaded to MinIO.")


def load_to_postgres(**kwargs):
    logging.info("Loading transformed data to Postgres...")
    pg_hook = PostgresHook(postgres_conn_id="postgres_default")
    df = pd.read_csv("/tmp/transformed_orders.csv")
    pg_hook.run("TRUNCATE TABLE orders_transformed;")
    for _, row in df.iterrows():
        insert_sql = """
        INSERT INTO orders_transformed(order_id, customer_id, amount, processed_timestamp)
        VALUES (%s, %s, %s, %s)
        """
        pg_hook.run(
            insert_sql,
            parameters=(
                row["order_id"],
                row["customer_id"],
                row["amount"],
                row["processed_timestamp"],
            ),
        )
    logging.info("Data loaded into Postgres.")


with DAG(
    dag_id="batch_ingestion_dag",
    default_args=default_args,
    schedule_interval="@daily",
    catchup=False,
) as dag:
    extract_task = PythonOperator(
        task_id="extract_mysql", python_callable=extract_data_from_mysql
    )

    validate_task = PythonOperator(
        task_id="validate_data", python_callable=validate_data_with_ge
    )

    load_to_minio_task = PythonOperator(
        task_id="load_to_minio", python_callable=load_to_minio
    )

    spark_transform_task = BashOperator(
        task_id="spark_transform",
        bash_command="spark-submit --master local[2] /opt/spark_jobs/spark_batch_job.py",
    )

    load_postgres_task = PythonOperator(
        task_id="load_to_postgres", python_callable=load_to_postgres
    )

    (
        extract_task
        >> validate_task
        >> load_to_minio_task
        >> spark_transform_task
        >> load_postgres_task
    )

In [ ]:
# Spark Batch Job (pipelines/spark/spark_batch_job.py) - current implementation excerpt
from pathlib import Path

batch_job_path = Path("pipelines/spark/spark_batch_job.py")
batch_job_code = batch_job_path.read_text(encoding="utf-8")

print(f"Loaded file: {batch_job_path}")
print("-" * 80)
print(batch_job_code[:7000])
print("\n... (truncated)")

In [ ]:
# Spark Streaming Job (spark/spark_streaming_job.py)
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
import logging
import great_expectations as ge

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

KAFKA_BROKER = "kafka:9092"
KAFKA_TOPIC = "sensor_readings"
MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
RAW_DATA_PATH = "s3a://raw-data/streaming_raw/"
ANOMALY_DATA_PATH = "s3a://processed-data/streaming_anomalies/"

POSTGRES_TABLE = "anomalies_stream"

schema = StructType(
    [
        StructField("event_id", StringType(), False),
        StructField("timestamp", LongType(), False),
        StructField("device_id", LongType(), False),
        StructField("reading_value", DoubleType(), False),
    ]
)


def validate_schema(df):
    logging.info("Validating streaming data schema with Great Expectations...")
    ge_df = ge.from_pandas(df.toPandas())
    result_event_id = ge_df.expect_column_values_to_not_be_null("event_id")
    if not result_event_id.success:
        raise ValueError("Validation failed: 'event_id' contains null values")
    result_timestamp = ge_df.expect_column_values_to_be_between(
        "timestamp", min_value=1
    )
    if not result_timestamp.success:
        raise ValueError("Validation failed: 'timestamp' contains invalid values")
    result_device = ge_df.expect_column_values_to_be_between("device_id", min_value=1)
    if not result_device.success:
        raise ValueError("Validation failed: 'device_id' contains invalid values")
    result_reading = ge_df.expect_column_values_to_be_between(
        "reading_value", min_value=0, max_value=100
    )
    if not result_reading.success:
        raise ValueError("Validation failed: 'reading_value' is out of expected range")
    logging.info("Streaming schema validation passed.")


def save_to_postgres(batch_df, batch_id):
    logging.info(f"Writing batch {batch_id} to PostgreSQL table {POSTGRES_TABLE}...")
    batch_df.write.format("jdbc").option(
        "url", "jdbc:postgresql://postgres:5432/processed_db"
    ).option("dbtable", POSTGRES_TABLE).option("user", "user").option(
        "password", "pass"
    ).option(
        "driver", "org.postgresql.Driver"
    ).mode(
        "append"
    ).save()
    logging.info(f"Batch {batch_id} written to PostgreSQL.")


def main():
    spark = (
        SparkSession.builder.appName("StreamingETL")
        .config("spark.jars.packages", "org.postgresql:postgresql:42.2.18")
        .getOrCreate()
    )

    spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", MINIO_ENDPOINT)
    spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", MINIO_ACCESS_KEY)
    spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", MINIO_SECRET_KEY)
    spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")

    logging.info(f"Starting Spark Structured Streaming from Kafka topic: {KAFKA_TOPIC}")
    df_raw = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKER)
        .option("subscribe", KAFKA_TOPIC)
        .option("startingOffsets", "latest")
        .load()
    )

    df_parsed = df_raw.select(
        from_json(col("value").cast("string"), schema).alias("json_data")
    ).select("json_data.*")

    df_clean = df_parsed.filter(
        col("event_id").isNotNull()
        & col("timestamp").isNotNull()
        & col("device_id").isNotNull()
        & col("reading_value").isNotNull()
    )

    validate_schema(df_clean)

    df_anomalies = df_clean.filter(col("reading_value") > 70.0)

    # Write raw data to MinIO
    df_clean.writeStream.format("parquet").option(
        "checkpointLocation", "/tmp/spark-checkpoints/raw"
    ).option("path", RAW_DATA_PATH).outputMode("append").start()

    # Write anomalies to MinIO
    df_anomalies.writeStream.format("parquet").option(
        "checkpointLocation", "/tmp/spark-checkpoints/anomalies"
    ).option("path", ANOMALY_DATA_PATH).outputMode("append").start()

    # Write anomalies to PostgreSQL using foreachBatch
    df_anomalies.writeStream.foreachBatch(
        lambda batch_df, batch_id: save_to_postgres(batch_df, batch_id)
    ).outputMode("append").start()

    logging.info("Streaming job started. Processing events in real time.")
    spark.streams.awaitAnyTermination()


if __name__ == "__main__":
    main()

In [ ]:
# Kafka Producer (pipelines/kafka/producer.py) - current implementation excerpt
from pathlib import Path

producer_path = Path("pipelines/kafka/producer.py")
producer_code = producer_path.read_text(encoding="utf-8")

print(f"Loaded file: {producer_path}")
print("-" * 80)
print(producer_code[:5000])
print("\n... (truncated)")

In [ ]:
# Governance / Data Lineage (pipelines/governance/atlas_stub.py) - current implementation excerpt
from pathlib import Path

governance_path = Path("pipelines/governance/atlas_stub.py")
governance_code = governance_path.read_text(encoding="utf-8")

print(f"Loaded file: {governance_path}")
print("-" * 80)
print(governance_code[:5000])
print("\n... (truncated)")

In [ ]:
# Monitoring setup (pipelines/monitoring/monitoring.py) - current implementation excerpt
from pathlib import Path

monitoring_path = Path("pipelines/monitoring/monitoring.py")
monitoring_code = monitoring_path.read_text(encoding="utf-8")

print(f"Loaded file: {monitoring_path}")
print("-" * 80)
print(monitoring_code[:5000])
print("\n... (truncated)")

In [ ]:
# MLflow tracking stub (pipelines/ml/mlflow_tracking.py) - current implementation excerpt
from pathlib import Path

mlflow_path = Path("pipelines/ml/mlflow_tracking.py")
mlflow_code = mlflow_path.read_text(encoding="utf-8")

print(f"Loaded file: {mlflow_path}")
print("-" * 80)
print(mlflow_code[:3000])
print("\n... (truncated)")

## Setup Instructions

1. **Clone the Repository**
   ```bash
   git clone https://github.com/paulchen8206/Modern-Enterprise-Data-Stack.git
   cd Modern-Enterprise-Data-Stack
   ```

2. **Start the Platform**
   Use Make as the standard interface:
   ```bash
   make up
   ```

3. **Verify Runtime Health**
   ```bash
   make ps
   make logs
   ```

4. **Access Core Services**
   - **Airflow UI:** [http://localhost:8080](http://localhost:8080)
   - **MinIO Console:** [http://localhost:9001](http://localhost:9001)
   - **Prometheus:** [http://localhost:9090](http://localhost:9090)
   - **Grafana:** [http://localhost:3000](http://localhost:3000)

5. **Run Pipelines**
   - Batch job:
     ```bash
     make run-batch-job
     ```
   - Streaming path:
     ```bash
     make run-kafka-producer
     make run-streaming-job
     ```

6. **Run Iceberg Demo (Optional)**
   ```bash
   make run-iceberg-demo
   ```
   This writes Spark batch output to an Iceberg table (`local.analytics.orders`).

7. **Validation & Formatting**
   ```bash
   make validate
   make format
   ```

In [ ]:
# Quick health snapshot: container status + core endpoint checks
import subprocess
from urllib.request import urlopen
from urllib.error import URLError, HTTPError

print("=== make ps ===")
ps = subprocess.run(
    ["make", "ps"],
    capture_output=True,
    text=True,
    check=False,
    timeout=120,
 )
print(ps.stdout if ps.stdout else ps.stderr)

print("\n=== endpoint checks ===")
endpoints = {
    "Airflow": "http://localhost:8080",
    "Prometheus": "http://localhost:9090",
    "Grafana": "http://localhost:3000",
    "MinIO Console": "http://localhost:9001",
}

for name, url in endpoints.items():
    try:
        with urlopen(url, timeout=5) as resp:
            print(f"[OK] {name}: HTTP {resp.status}")
    except HTTPError as e:
        print(f"[WARN] {name}: HTTP {e.code}")
    except URLError as e:
        print(f"[DOWN] {name}: {e.reason}")

## Example Applications

- **E-Commerce & Retail**
  - Real-time recommendations and fraud detection from event streams.
  - Batch sales processing for trend and demand analysis.

- **Financial Services**
  - Transaction anomaly detection and surveillance workflows.
  - Historical aggregation for risk modeling and compliance reporting.

- **Healthcare**
  - Real-time patient telemetry monitoring.
  - Batch clinical/operational analytics with governed lineage.

- **IoT & Manufacturing**
  - Predictive maintenance from sensor streams.
  - Process optimization using historical machine data.

- **Media & Social Platforms**
  - Sentiment and engagement analytics from high-volume streams.
  - Campaign quality and anomaly tracking with dashboard observability.

## Conclusion & Next Steps

This notebook now mirrors the **current repository state** by loading live source files and operational config directly from the project.

### Recommended Next Steps

- Run notebook code cells to inspect the latest compose and pipeline source definitions.
- Use `make run-iceberg-demo` and verify Iceberg write logs in Spark output.
- Extend Spark transformations and DAG orchestration for your production data contracts.
- Replace governance and ML stubs with real Atlas/OpenMetadata and MLflow integrations.
- Add notebook validation cells for smoke tests (service health, bucket checks, table row counts).

Happy building!